# Zadanie 5: programowanie genetyczne i regresja symboliczna

Termin realizacji: 18 maja 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zmodyfikuj przykład `pysr_demo.ipynb` tak, aby uczył się funkcji $f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$ której dziedziną jest $\mathbb{R}^6$. Uczenie ma się odbywać w oparciu o 200 wylosowanych z dziedziny próbek (między -5 a 5).
2. Zanotuj wzory trzech rozwiązań o najwyższej wartości `score` oraz rozwiązanie `best` dla następujących zestawów ustawień:
   1. `binary_operators=["+", "*"], unary_operators=["cos", "exp", "sin"], maxsize=20`,
   2. `binary_operators=["+", "*", "-", "^"], unary_operators=["cos", "exp", "sin", "log"], maxsize=30`, (dodaj ograniczenie dla argumentów operatora "^": [https://astroautomata.com/PySR/v1.5.9/options.html#constraining-use-of-operators](https://astroautomata.com/PySR/v1.5.9/options.html#constraining-use-of-operators).
   3. `binary_operators=["+", "*", "-", "^"], unary_operators=["exp", "sin"], maxsize=15`.
3. Powtórz eksperymenty z zadania na 3.0 po dodaniu szumu do próbek z funkcji $f$ (rozkład normalny o średniej 0 i odchyleniu standardowym 0.5)

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj do porównania dopasowanie oparte o próbki losowane w szerszym zakresie (między -15 a 15) oraz wyższy poziom szumu (odchylenie standardowe równe 2 oraz 5).

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zamień funkcję $f$ na $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$ gdzie $p(i)$ oznacza $i$-tą liczbę pierwszą. Uwzględnij `p` jako dodatkowy operator unarny analogicznie do przykładu "Julia packages and types" z notatnika `pysr_demo.ipynb`. Powtórz eksperymenty opisane w zadaniach na 3.0 i 4.0.


# Zadanie 1 (Na 3.0)

## Ekspeymenty bez sumu

In [1]:
import pysr
import sympy
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split

np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
y = 2.2 * np.sin(X[:, 0]+ 2*X[:,1]) - X[:,5]**2 - 3

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    verbosity=False,
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 12 (Score: 27.2876):
Wzór: x5*x5*(-1.0) + sin(x0 + x1 + x1)*2.2 - 3.0
------------------------------
Miejsce 4 (Score: 2.0312):
Wzór: x5*(-1.1931126)*x5
------------------------------
Miejsce 10 (Score: 1.1391):
Wzór: x5*x5*(-0.98993033) + sin(x0 + x1 + x1) - 3.0890245
------------------------------


x5*x5*(-1.0) + sin(x0 + x1 + x1)*2.2 - 3.0

In [8]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:96: UserWarning: You are using the `^` operator, but have not set up `constraints` for it. This may lead to overly complex expressions. One typical constraint is to use `constraints={..., '^': (-1, 1)}`, which will allow arbitrary-complexity base (-1) but only powers such as a constant or variable (1). For more tips, please see https://ai.damtp.cam.ac.uk/pysr/tuning/
  warnings.warn(


Miejsce 10 (Score: 13.6924):
Wzór: -x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002
------------------------------
Miejsce 4 (Score: 3.1427):
Wzór: -x5*x5 - 3.0129306
------------------------------
Miejsce 9 (Score: 1.1390):
Wzór: -x5*x5 + sin(x0 + x1 + x1) - 3.006972
------------------------------


-x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002

In [5]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin"],
    maxsize=15,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:96: UserWarning: You are using the `^` operator, but have not set up `constraints` for it. This may lead to overly complex expressions. One typical constraint is to use `constraints={..., '^': (-1, 1)}`, which will allow arbitrary-complexity base (-1) but only powers such as a constant or variable (1). For more tips, please see https://ai.damtp.cam.ac.uk/pysr/tuning/
  warnings.warn(


Miejsce 4 (Score: 3.2011):
Wzór: -x5*x5 - 3.0129473
------------------------------
Miejsce 10 (Score: 0.5006):
Wzór: -x5*x5 - sin(x1 + x1 - sin(x0)) - 2.9558642
------------------------------
Miejsce 7 (Score: 0.0223):
Wzór: -(x5*x5 - sin(x1*(-2.0612314))) - 3.0706282
------------------------------


-x5*x5 - sin(x1 + x1 - sin(x0)) - 2.9558642

## Eksperymenty z szumem

In [6]:
np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
y = 2.2 * np.sin(X[:, 0]+ 2*X[:,1]) - X[:,5]**2 - 3
noise = 0.5 * np.random.randn(200)
y_noised = y + noise

In [7]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    verbosity=False,
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 4 (Score: 2.0183):
Wzór: x5*x5*(-1.1879799)
------------------------------
Miejsce 10 (Score: 0.9422):
Wzór: x5*x5*(-0.9882137) + sin(x0 + x1 + x1) - 3.0375025
------------------------------
Miejsce 11 (Score: 0.7142):
Wzór: (x5*(-0.4506583)*x5 + sin(x0 + x1 + x1) - 1.3303696)*2.215451
------------------------------


(x5*(-0.4506583)*x5 + sin(x0 + x1 + x1) - 1.3303696)*2.215451

In [9]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:96: UserWarning: You are using the `^` operator, but have not set up `constraints` for it. This may lead to overly complex expressions. One typical constraint is to use `constraints={..., '^': (-1, 1)}`, which will allow arbitrary-complexity base (-1) but only powers such as a constant or variable (1). For more tips, please see https://ai.damtp.cam.ac.uk/pysr/tuning/
  warnings.warn(


Miejsce 4 (Score: 3.0386):
Wzór: -x5*x5 - 2.9472342
------------------------------
Miejsce 12 (Score: 0.7373):
Wzór: -(x5*x5 + cos(x0 - (-1.9809057)*(x1 + 0.7957469))*2.2223318) - 2.940425
------------------------------
Miejsce 11 (Score: 0.6359):
Wzór: -(x5*x5 + cos(-x0 + (x1 + 0.8079438)*(-1.9825742))) - 2.9447296
------------------------------


-(x5*x5 + cos(x0 - (-1.9809057)*(x1 + 0.7957469))*2.2223318) - 2.940425

In [10]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-", "^"],
    unary_operators=["exp", "sin"],
    maxsize=15,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\knotp\Documents\GitHub\MetodyOptymalizacji\zad5\.venv\Lib\site-packages\pysr\sr.py:96: UserWarning: You are using the `^` operator, but have not set up `constraints` for it. This may lead to overly complex expressions. One typical constraint is to use `constraints={..., '^': (-1, 1)}`, which will allow arbitrary-complexity base (-1) but only powers such as a constant or variable (1). For more tips, please see https://ai.damtp.cam.ac.uk/pysr/tuning/
  warnings.warn(


Miejsce 4 (Score: 3.0967):
Wzór: -x5*x5 - 2.9470928
------------------------------
Miejsce 9 (Score: 0.9574):
Wzór: -x5*x5 + sin(x0 - (-1.9819658)*x1) - 2.9439185
------------------------------
Miejsce 10 (Score: 0.7377):
Wzór: -x5*x5 + sin(x0 + x1*1.9806993)*2.222905 - 2.9401364
------------------------------


-x5*x5 + sin(x0 + x1*1.9806993)*2.222905 - 2.9401364